In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

# **Dataset & DataLoader**

In [30]:
from torch.utils.data import DataLoader
from torchvision.transforms import transforms

# image => scale (0, 1) => normalize (-1, 1)
transform = transforms.Compose([
    transforms.ToTensor(), # image -> pytorch tensor + sclae all the image
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

In [3]:
trainset = CIFAR10(root = "./data", train=True, download = True, transform = transform)
testset = CIFAR10(root = "./data", train=False, download=True, transform = transform)

100%|██████████| 170M/170M [25:41<00:00, 111kB/s]  


In [31]:
# from torch.utils.data import random_split

# train_size = 40000
# val_size = 10000

# train_dataset, val_dataset = random_split(
#     trainset,
#     [train_size, val_size]
# )

from torch.utils.data import random_split

train_size = int(0.8 * len(trainset))
val_size = len(trainset) - train_size

train_dataset, val_dataset = random_split(
    trainset,
    [train_size, val_size]
)

In [32]:
trainLoader = DataLoader(train_dataset, batch_size = 128, shuffle = True)
valLoader = DataLoader(val_dataset, batch_size = 128, shuffle = True)
testLoader = DataLoader (testset, batch_size = 128, shuffle = True)

# **Defn CNN**

In [33]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # 1st convolutional layer
            nn.Conv2d(3, 32, kernel_size = 3, padding=1), # input = 3, output = 32(create 32 feature map)
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # filter//kernel, stride
            #(16, 16, 32)

            # 2nd convolutional layer
            nn.Conv2d(32, 64, kernel_size = 3, padding = 1), # (16, 16, 32)
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # (8, 8, 64)

            # 3rd convolutional layer
            nn.Conv2d(64, 128, kernel_size = 3, padding = 1), # (8, 8, 64)
            nn.ReLU(),
            nn.MaxPool2d(2, 2) # (4, 4, 128)
        )

        self.fc_layers = nn.Sequential(
            # 1st layer
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            # output layer
            nn.Linear(256, 10)
        )


    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # used to flattern
        x = self.fc_layers(x)
        return x

In [34]:
# to use gpu

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [35]:
model = CNN().to(device)

In [36]:
# Loss and Optimizer

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# **Training CNN**

In [37]:
epochs = 100
best_val_loss = float("inf")
patience = 5
epoch_without_improvement = 0

for epoch in range(epochs):
    # training
    model.train()
    running_train_loss = 0.0

    for images, labels in trainLoader:
        # move data into GPU
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        #forward propagation
        outputs = model(images)
        loss = criterion(outputs, labels) # loss
        # back propagation
        loss.backward()
        optimizer.step() # update parameter

        running_train_loss += loss.item()

    epoch_train_loss = running_train_loss / len(trainLoader)

    # validation
    model.eval()
    running_val_loss = 0.0

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in valLoader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            # calculate val accuracy
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        epoch_val_loss = running_val_loss / len(testLoader)
        val_accuracy = correct / total

    print(f"epoch = {epoch+1}/{epochs}, loss = {epoch_train_loss}, val loss = {epoch_val_loss}, val_accuracy = {val_accuracy}")

    # early stop
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        # save the model
        torch.save(model.state_dict(), "best_model.pk")
        print("Validation loss improved. Saved Model")

    else:
        epoch_without_improvement += 1
        print(f"No improvenment: {epoch_without_improvement} / {patience}")

    if epoch_without_improvement >= patience:
        print("Early stop triggered!")
        break

epoch = 1/100, loss = 1.5294332302416476, val loss = 1.297369048565249, val_accuracy = 0.5328
Validation loss improved. Saved Model
epoch = 2/100, loss = 1.1239877061341137, val loss = 1.0332269638399534, val_accuracy = 0.6353
Validation loss improved. Saved Model
epoch = 3/100, loss = 0.9217234370045768, val loss = 0.9054399464703813, val_accuracy = 0.6834
Validation loss improved. Saved Model
epoch = 4/100, loss = 0.7874256454336758, val loss = 0.8441122332705727, val_accuracy = 0.7069
Validation loss improved. Saved Model
epoch = 5/100, loss = 0.6795018968490747, val loss = 0.780610562879828, val_accuracy = 0.7284
Validation loss improved. Saved Model
epoch = 6/100, loss = 0.5917371976108977, val loss = 0.77389664815951, val_accuracy = 0.7386
Validation loss improved. Saved Model
epoch = 7/100, loss = 0.5014082658976412, val loss = 0.7706712506994416, val_accuracy = 0.7467
Validation loss improved. Saved Model
epoch = 8/100, loss = 0.424549941628124, val loss = 0.7838548640661602, v

# **Evaluate Model**

In [38]:
total = 0
correct = 0

model.eval()

with torch.no_grad():
    for images, labels in testLoader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"total val: {total}, correct val: {correct}")
print("Accuracy: ", correct / total * 100)

total val: 10000, correct val: 7216
Accuracy:  72.16
